# Assignment 2b, multimodal emotion recognition on TESS

Trains and tests the speech only, text only and multimodal pipelines on a Colab GPU (an A100 if available, any GPU
works).

The code and the 2,800 TESS clips come from the private dataset repo `gsri-18/ltrc-a2-data`. Results and
checkpoints are uploaded to it as `results/A2b_results.zip` after each split, and running all cells again after a
lost session restores them and skips the finished runs.

1. **data** checks the 2,800 clips and writes the manifest and the two splits (word disjoint, cross speaker)
2. **features** caches the trimmed 16 kHz audio, log mel spectrograms, WavLM Base+ and BERT hidden states, and
   Whisper transcripts with their BERT features
3. **experiments** trains and tests 7 variants on 2 splits with 3 seeds (42 runs)
4. **analysis** writes the accuracy tables, t-SNE plots, confusion matrices and failure cases

**Unattended run.** Choose Run all, then answer the two prompts at the start (the Hugging Face token, or a Colab
secret named `HF_TOKEN`, and the Google Drive permission). After that nothing needs watching. Every cell and every
log line carries its IST time. When the last cell finishes, or a cell fails, or the run passes 6 h, the
notebook saves itself with all outputs, the logs and the results to Hugging Face (`gsri-18/ltrc-a2-data`,
`results/A2b/`) and Google Drive (`MyDrive/LTRC_Assignment2/A2b/`), and then disconnects and deletes the
runtime. Interrupting a cell by hand does not disconnect.

In [ ]:
TAG = "A2b"
DATA_REPO = "gsri-18/ltrc-a2-data"
MAX_HOURS = 6   # if still running after this, save everything and disconnect anyway

import datetime, json, os, shutil, threading, time

IST = datetime.timezone(datetime.timedelta(hours=5, minutes=30))
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
LOG_DIR = "/content/session_logs"
os.makedirs(LOG_DIR, exist_ok=True)
DRIVE_DIR = None        # MyDrive/LTRC_Assignment2/<TAG>, set below when Google Drive is mounted
ARTIFACTS = []          # result zips, copied to Drive as they are made


def now():
    return datetime.datetime.now(IST).strftime("%Y-%m-%d %H:%M:%S")


def log(msg):
    line = f"[{now()} IST] {msg}"
    print(line, flush=True)
    with open(f"{LOG_DIR}/session_log.txt", "a") as f:
        f.write(line + "\n")


def save_notebook():
    """The notebook as it is at this moment, with every output so far."""
    try:
        from google.colab import _message
        nb = _message.blocking_request("get_ipynb", request="", timeout_sec=120)["ipynb"]
        with open(f"{LOG_DIR}/{TAG}_executed.ipynb", "w") as f:
            json.dump(nb, f)
        log(f"exported the notebook with its outputs ({len(nb['cells'])} cells)")
    except Exception as e:
        log(f"could not export the notebook ({type(e).__name__} {e})")


def to_drive(path):
    if DRIVE_DIR and os.path.exists(path):
        try:
            if os.path.isdir(path):
                shutil.copytree(path, f"{DRIVE_DIR}/{os.path.basename(path)}_{STAMP}", dirs_exist_ok=True)
            else:
                shutil.copy(path, DRIVE_DIR)
        except Exception as e:
            log(f"Drive copy of {path} failed ({type(e).__name__} {e})")


def save_everything(status):
    with open(f"{LOG_DIR}/status.json", "w") as f:
        json.dump({"notebook": TAG, "status": status, "started": f"{START:%Y-%m-%d %H:%M:%S} IST",
                   "ended": f"{now()} IST", "minutes": round((time.time() - START.timestamp()) / 60, 1)}, f, indent=1)
    save_notebook()
    try:
        from huggingface_hub import HfApi
        HfApi().upload_folder(folder_path=LOG_DIR, path_in_repo=f"results/{TAG}/session_{STAMP}", repo_id=DATA_REPO,
                              repo_type="dataset", commit_message=f"{TAG} session {STAMP} {status}")
        log(f"saved the logs and the notebook to {DATA_REPO}/results/{TAG}/session_{STAMP}")
    except Exception as e:
        log(f"Hugging Face upload of the logs failed ({type(e).__name__} {e})")
    if DRIVE_DIR:
        for p in ARTIFACTS:
            to_drive(p)
        to_drive(LOG_DIR)
        try:
            from google.colab import drive
            drive.flush_and_unmount()
            log(f"saved everything to Google Drive {DRIVE_DIR}")
        except Exception as e:
            log(f"Drive flush failed ({type(e).__name__} {e})")


_done = threading.Lock()


def finish(status="completed"):
    """Save everything to Hugging Face and Drive, then disconnect and delete the runtime."""
    if not _done.acquire(blocking=False):
        return
    log(f"notebook {status} after {(time.time() - START.timestamp()) / 60:.1f} min, saving")
    save_everything(status)
    log("disconnecting and deleting the runtime")
    time.sleep(3)
    from google.colab import runtime
    runtime.unassign()


if "_session_hooks" not in globals():
    START = datetime.datetime.now(IST)
    STAMP = f"{START:%Y%m%d_%H%M%S}"

    def _cell_start(info):
        globals()["_cell_t0"] = time.time()
        globals()["_cell_first_line"] = (info.raw_cell.strip().splitlines() or [""])[0][:90]

    def _cell_end(result):
        took = time.time() - globals().get("_cell_t0", time.time())
        err = result.error_before_exec or result.error_in_exec
        status = "ok" if err is None else type(err).__name__
        print(f"[{now()} IST] finished in {took:.1f} s" + ("" if err is None else f", {status}"))
        path = f"{LOG_DIR}/cell_times.csv"
        new = not os.path.exists(path)
        with open(path, "a") as f:
            if new:
                f.write("finished_ist,seconds,status,first_line\n")
            f.write('{},{:.1f},{},"{}"\n'.format(now(), took, status, globals().get("_cell_first_line", "session setup").replace('"', "'")))
        if err is not None and not isinstance(err, KeyboardInterrupt):
            log(f"cell failed with {err!r}")
            finish("failed")

    get_ipython().events.register("pre_run_cell", _cell_start)
    get_ipython().events.register("post_run_cell", _cell_end)
    _watchdog = threading.Timer(MAX_HOURS * 3600, finish, args=(f"stopped by the {MAX_HOURS} h limit",))
    _watchdog.daemon = True
    _watchdog.start()
    _session_hooks = True
    log(f"session started, {TAG}, everything is saved and the runtime deleted at the end, after an error "
        f"or after {MAX_HOURS} h")

In [ ]:
import getpass, requests
from huggingface_hub import whoami


def can_write(token):
    """Asks the data repo how a small file would be uploaded, which needs write access and commits nothing."""
    r = requests.post(f"https://huggingface.co/api/datasets/{DATA_REPO}/preupload/main", timeout=30,
                      headers={"authorization": f"Bearer {token}"},
                      json={"files": [{"path": "results/write_check.txt", "size": 4, "sample": "dGVzdA=="}]})
    return r.status_code == 200


token = os.environ.get("HF_TOKEN", "")
if not token:
    try:
        from google.colab import userdata          # a Colab secret named HF_TOKEN, if there is one
        token = userdata.get("HF_TOKEN") or ""
    except Exception:
        token = ""
while not can_write(token):                        # checked now, not after an hour of training
    if token:
        log(f"this Hugging Face token cannot write to {DATA_REPO}")
    token = getpass.getpass("Hugging Face token with write access ").strip()
os.environ["HF_TOKEN"] = token
log(f"logged in to Hugging Face as {whoami()['name']}, with write access to {DATA_REPO}")

# Google Drive gets a second copy of everything. Declining the prompt keeps Hugging Face only.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = f"/content/drive/MyDrive/LTRC_Assignment2/{TAG}"
    os.makedirs(DRIVE_DIR, exist_ok=True)
    log(f"Google Drive mounted, copies go to MyDrive/LTRC_Assignment2/{TAG}")
except Exception as e:
    log(f"Google Drive not mounted ({type(e).__name__}), saving to Hugging Face only")

## Setup

In [ ]:
import sys, torch
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
log(f"Python {sys.version.split()[0]}, PyTorch {torch.__version__}, CUDA {torch.version.cuda}, {os.cpu_count()} CPU cores")

In [ ]:
import zipfile
from huggingface_hub import hf_hub_download

for name, dst in [("code.zip", "/content/project"), ("TESS.zip", "/content/project/Assignment_2b/data")]:
    path = hf_hub_download(DATA_REPO, name, repo_type="dataset", force_download=(name == "code.zip"))
    zipfile.ZipFile(path).extractall(dst)
    log(f"{name} -> {dst}")
log("code version " + open("/content/project/COMMIT").read().strip())
import transformers, librosa, sklearn
log(f"transformers {transformers.__version__}, librosa {librosa.__version__}, scikit-learn {sklearn.__version__}")

import re, subprocess, sys

PY = sys.executable  # Colab's Python, not the one in the aligner's environment


def run(cmd, keep=None, drop=None):
    """Run a shell command, print its filtered output with the time of each line, keep a copy in
    session_logs/run_output.txt and raise if it fails."""
    log(f"$ {cmd}")
    with open(f"{LOG_DIR}/run_output.txt", "a") as out:
        out.write(f"\n[{now()} IST] $ {cmd}\n")
        proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            for part in line.replace("\r", "\n").split("\n"):
                if part.strip() and (not keep or re.search(keep, part)) and not (drop and re.search(drop, part)):
                    stamped = f"[{datetime.datetime.now(IST):%H:%M:%S}] {part}"
                    print(stamped, flush=True)
                    out.write(stamped + "\n")
        code = proc.wait()
    if code != 0:
        raise RuntimeError(f"{cmd} failed with exit code {code}")


from huggingface_hub import HfApi


def upload_results():
    """Results/, the checkpoints and the Whisper transcripts, as results/A2b_results.zip in the data repo and in Drive."""
    out = "/content/A2b_results.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
        for top in ["Results", "checkpoints"]:
            for d, _, files in os.walk(f"/content/project/Assignment_2b/{top}"):
                for f in files:
                    p = os.path.join(d, f)
                    z.write(p, os.path.relpath(p, "/content/project/Assignment_2b"))
        asr = "/content/project/Assignment_2b/data/asr_whisper.json"
        if os.path.exists(asr):
            z.write(asr, "data/asr_whisper.json")
    if out not in ARTIFACTS:
        ARTIFACTS.append(out)
    to_drive(out)                                   # Drive first, so a failed upload loses nothing
    try:
        HfApi().upload_file(path_or_fileobj=out, path_in_repo="results/A2b_results.zip", repo_id=DATA_REPO,
                            repo_type="dataset", commit_message=f"A2b results {now()} IST")
        log(f"uploaded results/A2b_results.zip ({os.path.getsize(out) / 1e6:.1f} MB)"
            + (", copied to Drive" if DRIVE_DIR else ""))
    except Exception as e:
        log(f"Hugging Face upload failed ({type(e).__name__} {e})"
            + (", the zip is in Google Drive" if DRIVE_DIR else ", the run continues"))


# a previous session's results, so finished runs are skipped
try:
    zipfile.ZipFile(hf_hub_download(DATA_REPO, "results/A2b_results.zip", repo_type="dataset",
                                    force_download=True)).extractall("/content/project/Assignment_2b")
    log(f"restored {len(os.listdir('/content/project/Assignment_2b/Results/runs'))} finished runs")
except Exception as e:
    log(f"no earlier results ({type(e).__name__})")

## Data

In [ ]:
%cd /content/project/Assignment_2b
run(f"{PY} scripts/prepare_tess.py")

## Features

Trimmed 16 kHz audio and log mel spectrograms, then the frozen encoders (WavLM Base+ for speech, BERT for text).

In [ ]:
for kind in ["wav", "mel", "wavlm"]:
    run(f"{PY} -W ignore -m models.common.features --kind {kind}", drop=r"(?i)warn|unexpected|ignored")
run(f"{PY} -W ignore -m models.common.features --kind bert --text_source gt", drop=r"(?i)warn|unexpected|ignored|^- ")
!du -sh cache/*

Whisper transcripts of every clip, the second text source. The TESS transcript is the same carrier phrase for all
seven emotions, so this tests whether text that comes from speech recognition carries any emotional cue.

In [ ]:
run(f"{PY} -W ignore scripts/transcribe_whisper.py", drop=r"(?i)warn|deprecat")
run(f"{PY} -W ignore -m models.common.features --kind bert --text_source asr", drop=r"(?i)warn|unexpected|ignored|^- ")

## Experiments

Word disjoint split first (each word is only in train, val or test), then the cross speaker split (train on OAF,
test on YAF). Each run prints its size, best validation epoch and test result.

In [ ]:
run(f"{PY} scripts/run_experiments.py --split word --seeds 0 1 2")
upload_results()

In [ ]:
run(f"{PY} scripts/run_experiments.py --split cross_speaker --seeds 0 1 2")
upload_results()

## Analysis

In [ ]:
run(f"{PY} -W ignore scripts/analyse.py", drop=r"(?i)warn")
from IPython.display import Image, Markdown, display
for f in ["accuracy_tables.md", "per_emotion.md", "fusion_vs_speech.md", "failure_cases.md"]:
    display(Markdown(open(f"Results/{f}").read()))
for f in sorted(os.listdir("Results/plots")):
    print(f)
    display(Image(f"Results/plots/{f}"))
upload_results()

## Save and disconnect

Saves the executed notebook, the logs, the cell timings and the results to Hugging Face and Google Drive, then
disconnects and deletes this runtime.

In [ ]:
finish("completed")